## 🗑️ Eliminación y Recuperación de Managed y External Tables

Ahora que conocemos las diferencias entre las Managed Tables y las External Tables, surge una pregunta muy importante:

> 🤔 ¿Qué ocurre cuando eliminamos una tabla por error?

La respuesta dependerá del tipo de tabla que estemos utilizando.

---

* #### 📦 Managed Tables

Recordemos que una Managed Table almacena:

* ✅ Los metadatos en Unity Catalog
* ✅ Los datos físicos administrados por Databricks

Debido a ello, Databricks mantiene información suficiente para permitir su recuperación durante un período determinado. 

Ahora bien, para poder eliminar la tabla realizaremos:
```sql
DROP TABLE nombre_catalago.nombre_esquema.nombre_tabla;
```

Y para poder recuperar la tabla:
```sql
UNDROP TABLE nombre_catalago.nombre_esquema.nombre_tabla;
```

* ##### 🤯 ¿Qué ocurre internamente?

Cuando ejecutamos un `DROP TABLE` sobre una Managed Table, Databricks no elimina inmediatamente toda la información de forma permanente.

Durante un período de retención, la tabla puede recuperarse utilizando `UNDROP TABLE`.

Esto permite restaurar:
* ✅ Metadatos
* ✅ Datos físicos
* ✅ Definición original de la tabla

sin necesidad de recrearla manualmente.

---

#### 🌎 External Tables

Las External Tables funcionan de manera diferente. Recordemos que:

* ✅ Los metadatos se almacenan en Unity Catalog
* ✅ Los datos físicos permanecen en un almacenamiento externo
(S3, ADLS o GCS)

Ahora bien, para poder eliminar la tabla realizaremos:
```sql
DROP TABLE nombre_catalago.nombre_esquema.nombre_tabla;
```

Al ejecutar este comando:

* ❌ Se eliminan los metadatos registrados en Databricks
* ✅ Los archivos físicos permanecen intactos en la ubicación externa

A diferencia de las Managed Tables, las External Tables no pueden recuperarse mediante:

```sql
UNDROP TABLE
```

Para volver a utilizarlas es necesario registrar nuevamente la tabla apuntando a la ubicación donde continúan existiendo los datos.

En otras palabras:

* 🔄 Debemos recrear la definición de la tabla.
* 📂 Los datos físicos ya siguen existiendo.
* 🏗️ Solo necesitamos volver a registrar los metadatos.

[Crear-External-Tables](https://github.com/BrayanR03/Databricks-DE-2026/blob/main/06_TablesTypes.ipynb)

---

### 📊 Resumen

| Acción                                         | Managed Table | External Table |
| ---------------------------------------------- | ------------- | -------------- |
| DROP TABLE elimina metadatos                   | ✅             | ✅              |
| DROP TABLE elimina datos físicos               | ✅*            | ❌              |
| UNDROP TABLE disponible                        | ✅             | ❌              |
| Requiere recreación manual                     | ❌             | ✅              |
| Los datos permanecen en almacenamiento externo | ❌             | ✅              |

> 💡 La principal diferencia es que una Managed Table puede recuperarse mediante `UNDROP TABLE`, mientras que una External Table debe registrarse nuevamente utilizando la ubicación donde continúan almacenados los datos.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("07DropTables").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Managed Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed`

In [0]:
## ELIMINAR TABLA
spark.sql("DROP TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed")
print("Managed Table eliminada correctamente")

In [0]:
## RECUPERAR TABLA
spark.sql("UNDROP TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed")
print("Managed Table recuperada correctamente")

### 📂 External Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table`


In [0]:
## ELIMINAR TABLA
spark.sql("DROP TABLE catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table")
print("External Table eliminada correctamente")

In [0]:
## RECUPERAR TABLA (re-crearla)
spark.sql("""
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table
        USING CSV
        OPTIONS (
        header = 'true',
        inferSchema = 'true',
        delimiter = ','
        )
        LOCATION 's3://bucket-datasets-brayan/tips.csv';
          """)
print("External Table re-creada correctamente")